# Practical PyTorch · II — Part 6 — Companion Notebook

This goes with **"Did Your Model Actually Learn Anything? Evaluating Honestly"**. Run the cells top to bottom (Shift+Enter).

The goal: tell a model that *learned* apart from one that merely *memorized* — using train/validation/test splits, accuracy, F1, and the train-vs-validation gap. It runs on a plain CPU; no GPU needed.

## 0. Setup

We need the `evaluate` library for metrics, `datasets` for an easy split, and `scikit-learn` (some metrics use it under the hood).

In [ ]:
!pip install -q evaluate datasets scikit-learn

## 1. The metrics, in plain code

Let's start with the metrics themselves on a handful of made-up predictions, so the numbers are obvious before we wire anything into a `Trainer`.

Imagine a yes/no classifier. `1` = the thing we care about (say, fraud); `0` = ordinary.

In [ ]:
import numpy as np
import evaluate

acc       = evaluate.load("accuracy")
f1        = evaluate.load("f1")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")

# truth vs. a model's predictions
labels = [1, 0, 0, 1, 0, 1, 0, 0, 1, 0]
preds  = [1, 0, 0, 0, 0, 1, 0, 1, 1, 0]   # missed one '1', raised one false alarm

print(acc.compute(predictions=preds, references=labels))
print(precision.compute(predictions=preds, references=labels))
print(recall.compute(predictions=preds, references=labels))
print(f1.compute(predictions=preds, references=labels))

## 2. Why accuracy can lie

Now the famous trap. Build an *imbalanced* problem — 95% of examples are class `0` — and a lazy "model" that always guesses `0`. Watch accuracy look great while F1 exposes the truth.

In [ ]:
# 95 ordinary (0), 5 rare (1)
labels = [0] * 95 + [1] * 5
always_zero = [0] * 100   # the lazy model: never flags anything

print("accuracy:", acc.compute(predictions=always_zero, references=labels))
print("f1:      ", f1.compute(predictions=always_zero, references=labels))
# 95% accurate, and utterly useless — F1 is 0.0 because it caught zero of the rare class.

That's the whole lesson in two lines: a single accuracy number isn't the whole story. On imbalanced data, look at F1.

## 3. The compute_metrics function

The `Trainer` doesn't hand you clean predictions — it hands you **logits** (raw per-class scores) and the true labels. `np.argmax` turns each row of scores into a single predicted class, then we compute the metrics. This exact function plugs straight into the `Trainer`.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)   # highest-scoring class per example
    return {
        **acc.compute(predictions=preds, references=labels),
        **f1.compute(predictions=preds, references=labels, average="weighted"),
    }

# Sanity check with fake logits: 3 examples, 2 classes each.
fake_logits = np.array([[2.1, 0.3],    # predicts class 0
                        [0.1, 1.8],    # predicts class 1
                        [0.4, 2.2]])   # predicts class 1
fake_labels = np.array([0, 1, 0])      # last one is wrong
print(compute_metrics((fake_logits, fake_labels)))

## 4. Splitting data: train / validation / test

You can't grade on what you studied. Here's the one-liner split, on a small built-in dataset. We carve out a test set first, then split the rest into train and validation.

In [ ]:
from datasets import load_dataset

# A small, fast sentiment dataset (sentences labelled positive/negative).
raw = load_dataset("rotten_tomatoes")

# rotten_tomatoes already ships train/validation/test — the ideal three piles.
print(raw)
print("train:      ", len(raw["train"]))
print("validation: ", len(raw["validation"]))
print("test:       ", len(raw["test"]))

If a dataset only gives you one pile, you split it yourself — the `seed` makes it reproducible so everyone grades on the same questions:

In [ ]:
# Example of a manual split (we won't use this below, since rotten_tomatoes
# already has three piles — but this is the pattern when a dataset has only one).
split = raw["train"].train_test_split(test_size=0.2, seed=42)
print("train: ", len(split["train"]))
print("held out:", len(split["test"]))

## 5. Putting it together: fine-tune, then evaluate honestly

This builds on the Part 5 fine-tuning setup. We tokenize the text, fine-tune a small model for a single quick epoch, and — the new part — hand `compute_metrics` to the `Trainer` so it reports accuracy and F1 on held-out data, not just loss.

We use small subsets so it runs in a couple of minutes on CPU. A GPU (Runtime → Change runtime type → GPU) makes it snappier.

In [ ]:
!pip install -q transformers

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

# Keep it small and fast.
train_ds = raw["train"].shuffle(seed=42).select(range(1000)).map(tokenize, batched=True)
eval_ds  = raw["validation"].shuffle(seed=42).select(range(300)).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
args = TrainingArguments(
    output_dir="out",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",     # evaluate on the held-out set each epoch
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,   # <-- the new piece
)

trainer.train()

## 6. The honest report card

Now ask for the held-out numbers explicitly. These are the ones you can quote.

In [ ]:
metrics = trainer.evaluate()
print(metrics)
# e.g. {'eval_loss': ..., 'eval_accuracy': 0.8x, 'eval_f1': 0.8x, ...}

## 7. Read off overfitting vs. underfitting

Compare the score on the *training* data to the score on the *held-out* data. A big gap means overfitting (memorized); both low means underfitting (didn't learn enough); a small gap is healthy.

In [ ]:
# Score on the data the model learned from...
train_metrics = trainer.evaluate(eval_dataset=train_ds)

train_acc = train_metrics["eval_accuracy"]
val_acc   = metrics["eval_accuracy"]
gap = train_acc - val_acc

print(f"train accuracy:      {train_acc:.3f}")
print(f"validation accuracy: {val_acc:.3f}")
print(f"gap:                 {gap:.3f}")

if train_acc < 0.7 and val_acc < 0.7:
    print("-> both low: looks like UNDERFITTING (train longer / bigger model)")
elif gap > 0.15:
    print("-> big gap: looks like OVERFITTING (more data / fewer epochs)")
else:
    print("-> small gap: looks HEALTHY")

## 8. The final exam: touch the test set once

Everything above used train and validation. The **test** set has stayed sealed. We touch it exactly once, now, for the number we'd actually report — and never tune against it.

In [ ]:
test_ds = raw["test"].shuffle(seed=42).select(range(300)).map(tokenize, batched=True)
test_metrics = trainer.evaluate(eval_dataset=test_ds)

print("FINAL (test set) accuracy:", round(test_metrics["eval_accuracy"], 3))
print("FINAL (test set) f1:      ", round(test_metrics["eval_f1"], 3))

That's the honest report card: a number from data the model never learned from, and never got tuned against. The rest is discipline — split before you train, watch the train-vs-validation gap, reach for F1 when classes are lopsided, and keep the test set sealed until the end.

**Next:** Part 7 — Parameter-Efficient Fine-Tuning (LoRA & PEFT), where you fine-tune big models without the big bill.